# Camada GOLD - Analise base de dados "Viagens a Serviço do Portal da Transparência do Governo Federal"

Este notebook le a camada **SILVER** (ja limpa e tipada) e responde 3 perguntas
de negocio. Para cada uma: a consulta SQL, a tabela com o resultado e um grafico.

**Análises realizadas**

| Pergunta                                              |                                                                
| ----------------------------------------------------- |
| **Os 5 órgãos com maior custo total?**                
| **Os 3 destinos com maior custo médio por viagem?**       
| **A viagem de maior duração e seu custo total?** 



## Conexao com o banco 


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

import sys
from pathlib import Path

# Pasta raiz do arquivo
raiz = Path().resolve().parent
if str(raiz) not in sys.path:
    sys.path.append(str(raiz))
import banco

conexao = banco.conectar()

def consultar(sql):
    """Roda um SELECT e devolve o resultado como DataFrame do pandas."""
    return pd.read_sql(sql, conexao)

def reais(valor):
    """Formata um numero como moeda brasileira: 1234.5 -> 'R$ 1.234,50'."""
    texto = f'{valor:,.2f}'
    return 'R$ ' + texto.replace(',', 'X').replace('.', ',').replace('X', '.')

print('Conectado ao MySQL com sucesso.')

## Análise 1 – Os 5 órgãos com maior custo total

Soma total dos gastos com as viagens agrupados pelo órgão superior da viagem e ordenado de forma decrescente (maior ao menor).

Através do gráfico de barras horizontais, é possível visualizar a comparação entre categorias.

In [ ]:
#Query SQL

SQL_A1 = """
SELECT
    nome_orgao_superior,
    ROUND(SUM(valor_total),2) AS custo_total
FROM silver_viagem
GROUP BY nome_orgao_superior
ORDER BY custo_total DESC
LIMIT 3;
"""
dataframe_1 = consultar(SQL_A1)
df1 = dataframe_1.copy()
df1["custo_total"] = df1["custo_total"].apply(reais)
display(df1)


#Plotagem do gráfico
plt.figure(figsize=(16,8))
plt.grid(axis='x', linestyle='--', alpha=0.4)

plt.barh(
    dataframe_1["nome_orgao_superior"],
    dataframe_1["custo_total"],
    color='darkred'
)

# Adiciona o valor ao final de cada barra
for i, valor in enumerate(dataframe_1["custo_total"]):
    plt.text(
        valor,                 # posição X
        i,                     # posição Y
        "  " + reais(valor),   # texto
        va='center',
        fontsize=10
    )

plt.title("Órgãos com maior custo total (TOP 3)")
plt.xlabel("Custo total (R$)")
plt.ylabel("Órgão Superior")

plt.gca().invert_yaxis()

#Remove bordas dos eixos
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)


# Desativa a notação científica 
plt.gca().ticklabel_format(style='plain', axis='x')

# Formata o eixo em milhões
plt.gca().xaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f'{x/1_000_000:.0f} mi')
)

plt.tight_layout()
plt.show()



## Análise 2 – Os 3 destinos com maior custo médio por viagem?
A consulta contabiliza o TOP 3 destinos com maior custo por viagem cruzando dados das duas tabelas (viagem e trecho)
O gráfico de pirulito diversifica e como há poucas categorias, esse tipo de visualização facilita a leitura dos valores.

In [ ]:
#Query SQL
SQL_A2 = """
SELECT
    t.destino_cidade,
    COUNT(DISTINCT v.id_viagem) AS qtd_viagens,
    ROUND(AVG(v.valor_total),2) AS custo_medio
FROM silver_viagem v
INNER JOIN silver_trecho t
    ON v.id_viagem = t.id_viagem
GROUP BY t.destino_cidade
ORDER BY custo_medio DESC
LIMIT 3;
"""

df2 = consultar(SQL_A2)
display(df2)

#Plotagem do gráfico
plt.figure(figsize=(10,5))

# Linhas
plt.hlines(
    y=df2["destino_cidade"],
    xmin=0,
    xmax=df2["custo_medio"],
    color="gray",
    linewidth=2
)

# Pontos
plt.scatter(
    df2["custo_medio"],
    df2["destino_cidade"],
    s=180
)

# Valores
for i, valor in enumerate(df2["custo_medio"]):
    plt.text(
        valor,
        i,
        "  " + reais(valor),
        va="center",
        fontsize=10
    )

plt.title("Top 3 destinos com maior custo médio por viagem")
plt.xlabel("Custo médio (R$)")
plt.ylabel("Destino")

plt.gca().invert_yaxis()

# Grade
plt.grid(axis='x', linestyle='--', alpha=0.4)

# Remove bordas
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['left'].set_visible(False)

plt.tight_layout()
plt.show()


## Análise 3 – Qual a viagem de maior duração e seu custo total?

Esta consulta filtra a quantidade de dias de cada viagem, ordenando por aquela de maior duração e traz seu respectivo valor.
O gráfico de barras vertical foi utilizado, apesar de não ter comparação por ser somente um registro, permite a visualização perante os eixos de valor e duração.

In [ ]:
#Query SQL
SQL_A3 = """
SELECT
    v.id_viagem,
    v.nome_orgao_superior,
    CONCAT(
        MIN(t.origem_uf),
        ' → ',
        GROUP_CONCAT(t.destino_uf ORDER BY t.sequencia_trecho SEPARATOR ' → ')
    ) AS trajeto,
    v.duracao_dias,
    v.valor_total
FROM silver_viagem v
INNER JOIN silver_trecho t
    ON v.id_viagem = t.id_viagem
GROUP BY
    v.id_viagem,
    v.nome_orgao_superior,
    v.duracao_dias,
    v.valor_total
ORDER BY
    v.duracao_dias DESC,
    v.valor_total DESC
LIMIT 1;
"""

df3 = consultar(SQL_A3)
display(df3)

plt.figure(figsize=(6,6))

plt.bar(
    "Viagem mais longa",
    df3["valor_total"].iloc[0]
)

# Exibe o valor acima da barra
plt.text(
    0,
    df3["valor_total"].iloc[0],
    f"{reais(df3['valor_total'].iloc[0])}\n{df3['duracao_dias'].iloc[0]} dias",
    ha="center",
    va="bottom",
    fontsize=10
)

plt.title("Viagem de maior duração e seu custo total")
plt.ylabel("Custo Total (R$)")

# Remove bordas
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Grade horizontal
plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

--
## Camada GOLD Agregada

**Tabela GOLD agregada:** Construção de uma camada GOLD com JOIN + GROUP BY com fucoes de
agregacao (`SUM`, `AVG`, `COUNT`, como tabela e como VIEW)


Criada utilizando JOIN + GROUP BY e fica preparada para responder diversas perguntas de negócio.

| Pergunta                                                          |                                                                             
| -----------------------------------------------------             | 
| **Qual o tipo de pagamento com maior valor médio?**               | 
| **Qual o meio de transporte mais usado nos trechos?**             | 
| **Qual UF de destino aparece em mais trechos?**                   | 







In [ ]:
# TAREFA 1: Criar camadas Gold com JOIN, como TABELA e como VIEW.

# ==============================================================================
#  1. GOLD - VISÃO GERAL DE VIAGENS 
#  Analisar custos totais, durações, motivos, órgãos e perfis de viajantes.
# ==============================================================================

SQL_AGREGADA_1 = """

DROP TABLE IF EXISTS gold_tb_analise_viagens;
CREATE TABLE gold_tb_analise_viagens AS
SELECT 
    v.id_viagem,
    v.num_proposta,
    v.situacao,
    v.viagem_urgente,
    v.cod_orgao_superior,
    v.nome_orgao_superior,
    v.nome_viajante,
    v.cargo,
    v.data_inicio,
    v.data_fim,
    v.duracao_dias,
    v.motivo,
    v.destinos AS itinerario_completo,
    v.valor_diarias,
    v.valor_passagens,
    v.valor_devolucao,
    v.valor_outros_gastos,
    v.valor_total AS custo_total_viagem
FROM silver_viagem v;
"""
SQL_AGREGADA_VIEW_1 = """
DROP VIEW IF EXISTS vw_gold_analise_viagens;
CREATE VIEW vw_gold_analise_viagens AS SELECT * FROM gold_tb_analise_viagens;

"""

# ==============================================================================
# 2. GOLD - VISÃO DE LOGÍSTICA E TRECHOS 
# AnalisE focada em rotas (origem/destino) e meios de transporte utilizados
# ==============================================================================

SQL_AGREGADA_2 = """
DROP TABLE IF EXISTS gold_tb_analise_logistica;
CREATE TABLE gold_tb_analise_logistica AS
SELECT 
    t.id_trecho,
    t.id_viagem,
    t.sequencia_trecho,
    t.origem_data,
    t.origem_uf,
    t.origem_cidade,
    t.destino_data,
    t.destino_uf,
    t.destino_cidade,
    t.meio_transporte,
    t.numero_diarias,
    -- Contexto da viagem acoplado para permitir filtros cruzados diretos
    v.nome_orgao_superior,
    v.cargo,
    v.situacao,
    -- Cruzamento com a tabela de passagens para capturar o custo do bilhete se houver
    p.valor_passagem,
    p.taxa_servico
FROM silver_trecho t
JOIN silver_viagem v ON t.id_viagem = v.id_viagem
LEFT JOIN silver_passagem p ON t.id_viagem = p.id_viagem 
    AND t.origem_uf = p.uf_origem_ida 
    AND t.destino_uf = p.uf_destino_ida;

"""

SQL_AGREGADA_VIEW_2 = """
DROP VIEW IF EXISTS vw_gold_analise_logistica;
CREATE VIEW vw_gold_analise_logistica AS SELECT * FROM gold_tb_analise_logistica;
"""

# ==============================================================================
#   3. GOLD - INFORMAÇÕES FINANCEIRAS (PAGAMENTOS) 
# Cruzamento de dados sobre quem pediu a viagem (órgão superior) e pra quem realmente o gasto foi debitado (UG).
# ==============================================================================

SQL_AGREGADA_3 = """
DROP TABLE IF EXISTS gold_tb_analise_pagamentos;
CREATE TABLE gold_tb_analise_pagamentos AS
SELECT 
    pag.id_pagamento,
    pag.id_viagem,
    pag.num_proposta,
    v.nome_orgao_superior AS orgao_solicitante,
    pag.nome_orgao_pagador,
    pag.nome_ug_pagadora,
    pag.tipo_pagamento,
    pag.valor AS valor_pago,
    v.situacao AS situacao_viagem
FROM silver_pagamento pag
JOIN silver_viagem v ON pag.id_viagem = v.id_viagem;
"""

SQL_AGREGADA_VIEW_3 = """
DROP VIEW IF EXISTS vw_gold_analise_pagamentos;
CREATE VIEW vw_gold_analise_pagamentos AS SELECT * FROM gold_tb_analise_pagamentos;

"""





In [ ]:
# TAREFA 1: Criar camadas Gold com JOIN, como TABELA e como VIEW.

# ==============================================================================
#  1. GOLD - VISÃO GERAL DE VIAGENS 
#  Analisar custos totais, durações, motivos, órgãos e perfis de viajantes.
# ==============================================================================

SQL_DROP_TABELAS = """
DROP TABLE IF EXISTS gold_tb_analise_viagens, gold_tb_resumo_destinos, gold_tb_duracao_viagens, gold_tb_uso_transportes ;

"""
SQL_DROP_VIEWS = """ 
DROP VIEW IF EXISTS vw_gold_analise_viagens, vw_gold_resumo_destinos, vw_gold_duracao_viagens,vw_gold_uso_transportes;
"""

SQL_AGREGADA_1 = """
CREATE TABLE gold_tb_analise_viagens AS
SELECT 
    v.id_viagem,
    v.num_proposta,
    v.situacao,
    v.viagem_urgente,
    v.cod_orgao_superior,
    v.nome_orgao_superior,
    v.nome_viajante,
    v.cargo,
    v.data_inicio,
    v.data_fim,
    v.duracao_dias,
    v.motivo,
    v.destinos AS itinerario_completo,
    v.valor_diarias,
    v.valor_passagens,
    v.valor_devolucao,
    v.valor_outros_gastos,
    v.valor_total AS custo_total_viagem
FROM silver_viagem v;
"""
SQL_AGREGADA_VIEW_1 = """
CREATE VIEW vw_gold_analise_viagens AS SELECT * FROM gold_tb_analise_viagens;
"""

# ==============================================================================
# 2. Custo por Destino: Destinos com maior custo médio por viagem
# ==============================================================================

SQL_AGREGADA_2 = """
CREATE TABLE gold_tb_resumo_destinos AS
SELECT 
    t.destino_uf,
    t.destino_cidade,
    AVG(v.valor_total) AS custo_medio_viagem,
    COUNT(DISTINCT v.id_viagem) AS qtd_viagens_recebidas
FROM silver_trecho t
JOIN silver_viagem v ON t.id_viagem = v.id_viagem
WHERE t.destino_cidade IS NOT NULL
GROUP BY t.destino_uf, t.destino_cidade;
"""
SQL_AGREGADA_VIEW_2 = """
CREATE VIEW vw_gold_resumo_destinos AS SELECT * FROM gold_tb_resumo_destinos;
"""

# ==============================================================================
# 3. Duração e Escalas de Viagem: Viagem de maior duração e seu custo) 
# ==============================================================================

SQL_AGREGADA_3 = """
CREATE TABLE gold_tb_duracao_viagens AS
SELECT 
    v.id_viagem,
    v.nome_viajante,
    v.duracao_dias,
    v.valor_total AS custo_total_viagem,
    COUNT(t.id_trecho) AS total_trechos_na_viagem
FROM silver_viagem v
JOIN silver_trecho t ON v.id_viagem = t.id_viagem
GROUP BY v.id_viagem, v.nome_viajante, v.duracao_dias, v.valor_total;
"""

SQL_AGREGADA_VIEW_3 = """
CREATE VIEW vw_gold_duracao_viagens AS SELECT * FROM gold_tb_duracao_viagens;
"""

#==============================================================================
# 4. Transportes: Meio de transporte mais usado nos trechos
# ==============================================================================
SQL_AGREGADA_4 = """
CREATE TABLE gold_tb_uso_transportes AS
SELECT 
    t.meio_transporte,
    v.nome_orgao_superior,
    COUNT(t.id_trecho) AS qtd_trechos_percorridos
FROM silver_trecho t
JOIN silver_viagem v ON t.id_viagem = v.id_viagem
WHERE t.meio_transporte IS NOT NULL
GROUP BY t.meio_transporte, v.nome_orgao_superior;
"""
SQL_AGREGADA_VIEW_4 = """
CREATE VIEW vw_gold_uso_transportes AS SELECT * FROM gold_tb_uso_transportes;
"""
#==============================================================================
# 4. utilização de Transportes: Meio de transporte mais usado nos trechos
# ==============================================================================
SQL_AGREGADA_5 = """
CREATE TABLE gold_tb_financeiro_pagamentos AS
SELECT 
    tipo_pagamento,
    AVG(valor) AS valor_medio_pagamento,
    SUM(valor) AS valor_total_pago,
    COUNT(id_pagamento) AS qtd_transacoes
FROM silver_pagamento
GROUP BY tipo_pagamento;
"""

SQL_AGREGADA_VIEW_5 = """
CREATE VIEW vw_gold_financeiro_pagamentos AS SELECT * FROM gold_tb_financeiro_pagamentos;
"""

def camada_gold():

    print("=== CRIAÇÃO DAS TABELAS CAMADA GOLD ===")

    try:

        conexao = banco.conectar()
        print(" ==== [1/3] Apagando tabelas caso já existam ====")
        banco.executar(conexao, SQL_DROP_TABELAS)
        print(" ==== [1/3] Tabelas apagadas OK ====")

        print(" ==== [2/3] Apagando views caso já existam ====")
        banco.executar(conexao, SQL_DROP_VIEWS)
        print(" ==== [2/3] Views apagadas OK ====")

        print(" ==== [3/3] Criando Tabelas e Views ====")
               
        banco.executar(conexao, SQL_AGREGADA_1)
        print("  Tabela gold_tb_analise_viagens OK")
        banco.executar(conexao, SQL_AGREGADA_VIEW_1)        
        print("  View vw_gold_analise_viagens OK")

        banco.executar(conexao, SQL_AGREGADA_2)
        print("  Tabela gold_tb_analise_logistica OK")
        banco.executar(conexao, SQL_AGREGADA_VIEW_2)        
        print("  View vw_gold_analise_logistica OK")

        banco.executar(conexao, SQL_AGREGADA_3)
        print("  Tabela gold_tb_analise_pagamentos OK")
        banco.executar(conexao, SQL_AGREGADA_VIEW_3)       
        print("  View vw_gold_analise_pagamentos OK")

        banco.executar(conexao, SQL_AGREGADA_4)
        print("  Tabela gold_tb_analise_pagamentos OK")
        banco.executar(conexao, SQL_AGREGADA_VIEW_4)       
        print("  View vw_gold_analise_pagamentos OK")

        conexao.close()

        print("=== CRIAÇÃO CAMADA GOLD AGREGADA CONCLUÍDA ===")

    except Exception as erro:

        print("[ERRO]", erro)
        raise


camada_gold()

In [ ]:
def camada_gold():

    print("=== CRIAÇÃO DAS TABELAS CAMADA GOLD ===")

    try:

        conexao = banco.conectar()

        banco.executar(conexao, SQL_AGREGADA_1)
        banco.executar(conexao, SQL_AGREGADA_VIEW_1)
        print("  Tabela gold_tb_analise_viagens OK")
        print("  View vw_gold_analise_viagens OK")

        banco.executar(conexao, SQL_AGREGADA_2)
        banco.executar(conexao, SQL_AGREGADA_VIEW_2)
        print("  Tabela gold_tb_analise_logistica OK")
        print("  View vw_gold_analise_logistica OK")

        banco.executar(conexao, SQL_AGREGADA_3)
        banco.executar(conexao, SQL_AGREGADA_VIEW_3)
        print("  Tabela gold_tb_analise_pagamentos OK")
        print("  View vw_gold_analise_pagamentos OK")

        conexao.close()

        print("=== CRIAÇÃO CAMADA GOLD AGREGADA CONCLUÍDA ===")

    except Exception as erro:

        print("[ERRO]", erro)
        raise


camada_gold()

ANÁLISE 
2. Os 3 destinos com maior custo médio por viagem?
3. A viagem de maior duração e seu custo total?
4. Qual o tipo de pagamento com maior valor médio?
7. Qual órgão pagou mais no total?

In [ ]:
# Encerra a conexao com o banco
conexao.close()
print('Conexao encerrada.')